In [ ]:
from IPython.display import Markdown, display

from invoiceops.demo_paths import canonical_demo_root, initialize_demo_root, validate_demo_root
from invoiceops.demo_reset import reset_local_demo

# Side-effect contract: resource=canonical var/local-demo; condition=explicit initialization and confirmation; idempotency=reset recreates SQLite and bootstrap recreates the dataset; recovery=rerun dry-run, initialize, then bootstrap.
DEMO_ROOT = canonical_demo_root()
INITIALIZE_DEMO_ROOT = False  # Cambia a True una vez si el marcador todavía no existe.
RESET_CONFIRMATION = ""  # Escribe exactamente --confirm-reset-local-demo para habilitar el reset.

try:
    validate_demo_root(DEMO_ROOT, require_marker=True)
    marker_status = "El marcador de propiedad ya existe; se reutilizará."
except ValueError:
    marker_status = "No existe un marcador de propiedad utilizable todavía."

if INITIALIZE_DEMO_ROOT:
    try:
        initialize_demo_root(DEMO_ROOT)
        marker_status = "Marcador de propiedad inicializado; ya puedes confirmar el reset."
    except ValueError as error:
        marker_status = f"No se inicializó el marcador: {error}"

if RESET_CONFIRMATION == "--confirm-reset-local-demo":
    try:
        validate_demo_root(DEMO_ROOT, require_marker=True)
    except ValueError:
        marker_status += " El reset no se ejecutó: activa INITIALIZE_DEMO_ROOT y vuelve a ejecutar esta celda."
    else:
        reset_local_demo(DEMO_ROOT, confirmed=True)
        marker_status += " Reset confirmado: SQLite fue recreada; ejecuta bootstrap para recrear el dataset."

display(Markdown(
    "## Inicio limpio de Notebook 01\n\n"
    "\n"
    f"**Estado del marcador:** {marker_status}\n\n"
    "1. **Inspeccionar.** `RESET_CONFIRMATION` queda vacío por defecto. Desde la raíz, `uv run python scripts/reset_demo.py --demo-root var/local-demo` es un *dry run* que lista exactamente las rutas y no cambia archivos.\n"
    "2. **Inicializar.** Si el estado indica que falta el marcador, cambia `INITIALIZE_DEMO_ROOT` a `True` y ejecuta esta celda una vez. Inicializar solo crea o reutiliza el marcador de propiedad de la raíz canónica `var/local-demo`; no resetea ni inicia MLflow.\n"
    "3. **Reset opcional.** Solo con el marcador presente, escribe exactamente `--confirm-reset-local-demo` en `RESET_CONFIRMATION` y vuelve a ejecutar la celda. El reset borra `invoiceops.db`, `invoiceops.db-shm`, `invoiceops.db-wal`, `mlflow.db`, `mlflow-artifacts/`, `notebook-state/state.json` y el dataset canónico `data/invoice-risk-v1` dentro de `var/local-demo`; recrea SQLite, pero no el dataset. No toca `data/` en la raíz, código, servicios remotos ni secretos.\n"
    "4. **Bootstrap separado.** Tras un reset, inicia MLflow y ejecuta desde la raíz `uv run python scripts/bootstrap_local_demo.py --db-path var/local-demo/invoiceops.db`. Bootstrap recrea/reutiliza el dataset canónico aislado y prepara SQLite/MLflow; no confirma ni ejecuta el reset. Configura `INVOICEOPS_DB_PATH=var/local-demo/invoiceops.db` antes de continuar.\n"
    "5. **Dataset.** La siguiente celda crea o valida solamente `var/local-demo/data/invoice-risk-v1/` con la seed, versión, metadatos y hashes declarados.\n"
    "5. **Siguiente paso.** Completa este notebook y continúa con `02_models_and_metrics.ipynb`.\n"
))


# Datos y baseline

Este notebook presenta el contrato de datos de InvoiceOps y dos baselines sobre un conjunto sintético determinista. Usa la raíz canónica aislada del demo: la API pública del proyecto crea o valida el dataset antes de usarlo.

**Linaje canónico.** Recurso: `var/local-demo/data/invoice-risk-v1/`. La API valida metadatos y hashes de cada split antes de reutilizarlo; si falta, lo crea con la seed y versión declaradas. Recuperación: usa el reset canónico seguido de bootstrap; no cambies seed, versión ni archivos a mano.

In [ ]:
import csv
import json

import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from invoiceops.demo_paths import canonical_demo_root
from invoiceops.ml.data import TARGET
from invoiceops.ml.features import MODEL_FEATURES
from invoiceops.ml.train import ensure_canonical_dataset

DATASET_ROOT = canonical_demo_root() / "data"
dataset_dir, _ = ensure_canonical_dataset(output_root=DATASET_ROOT)

with (dataset_dir / "metadata.json").open(encoding="utf-8") as file:
    metadata = json.load(file)


def load_split(name):
    with (dataset_dir / name).open(newline="", encoding="utf-8") as file:
        return list(csv.DictReader(file))


train_rows = load_split("train.csv")
validation_rows = load_split("validation.csv")
test_rows = load_split("test.csv")

split_ranges = {
    split_name: (min(row["submitted_at"] for row in rows), max(row["submitted_at"] for row in rows))
    for split_name, rows in {
        "train": train_rows,
        "validation": validation_rows,
        "test": test_rows,
    }.items()
}

metadata_columns = ["invoice_id", "submitted_at"]
expected_columns = [*metadata_columns, *MODEL_FEATURES, TARGET]
assert len(MODEL_FEATURES) == 8
assert list(train_rows[0]) == expected_columns
assert metadata["target"] == TARGET

## Split temporal

Los registros se ordenan por `submitted_at`: train contiene el pasado y validation un periodo posterior. No se mezclan aleatoriamente, porque eso filtraría información futura. Mostramos el rango de `test.csv` solo para comprobar su posición futura: no se usa para fit, métricas, selección ni decisiones. El futuro no debe contaminar el entrenamiento.

In [ ]:
target_distribution = {
    split_name: {
        "no_manual_review": sum(row[TARGET] == "False" for row in rows),
        "manual_review": sum(row[TARGET] == "True" for row in rows),
    }
    for split_name, rows in {"train": train_rows, "validation": validation_rows}.items()
}

for split_name, counts in target_distribution.items():
    total = sum(counts.values())
    start, end = split_ranges[split_name]
    print(
        f"{split_name:10} {start} to {end}  no_manual_review={counts['no_manual_review']:5} ({counts['no_manual_review'] / total:.1%})  manual_review={counts['manual_review']:5} ({counts['manual_review'] / total:.1%})"
    )

test_start, test_end = split_ranges["test"]
print(
    f"{'test':10} {test_start} to {test_end}  range only; not used for fit, metrics, selection, or decisions"
)

labels = ["No manual review", "Manual review"]
train_counts = [
    target_distribution["train"]["no_manual_review"],
    target_distribution["train"]["manual_review"],
]
validation_counts = [
    target_distribution["validation"]["no_manual_review"],
    target_distribution["validation"]["manual_review"],
]
positions = range(len(labels))
plt.bar([position - 0.2 for position in positions], train_counts, width=0.4, label="train")
plt.bar(
    [position + 0.2 for position in positions], validation_counts, width=0.4, label="validation"
)
plt.xticks(list(positions), labels)
plt.ylabel("Invoices")
plt.title("Target distribution by temporal split")
plt.legend()
plt.show()

## Baselines

Primero medimos una predicción constante de 0 (`False`). Después usamos `DummyClassifier(strategy='most_frequent')`, entrenado solamente con train. Ambos se evalúan sobre validation con accuracy, precision, recall y F1.

In [ ]:
def feature_matrix(rows):
    return [[row[feature] for feature in MODEL_FEATURES] for row in rows]


y_train = [row[TARGET] == "True" for row in train_rows]
y_validation = [row[TARGET] == "True" for row in validation_rows]
X_train = feature_matrix(train_rows)
X_validation = feature_matrix(validation_rows)

zero_predictions = [False] * len(y_validation)
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_predictions = dummy.predict(X_validation)


def metrics_for(predictions):
    return {
        "accuracy": accuracy_score(y_validation, predictions),
        "precision": precision_score(y_validation, predictions, zero_division=0),
        "recall": recall_score(y_validation, predictions, zero_division=0),
        "f1": f1_score(y_validation, predictions, zero_division=0),
    }


baseline_metrics = {
    "constant_zero": metrics_for(zero_predictions),
    "dummy_most_frequent": metrics_for(dummy_predictions),
}
for baseline_name, values in baseline_metrics.items():
    print(baseline_name)
    print("  " + "  ".join(f"{name}={value:.3f}" for name, value in values.items()))

## Matriz de confusión

Las filas son valores reales y las columnas son predicciones. Un falso positivo (FP) envía una factura segura a revisión manual; un falso negativo (FN) deja pasar una factura que requería revisión.

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_validation, dummy_predictions, labels=[False, True]).ravel()
print("                 Predicted no review  Predicted manual review")
print(f"Real no review  TN={tn:4}                 FP={fp:4}")
print(f"Real manual     FN={fn:4}                 TP={tp:4}")

### Pregunta

Una accuracy alta, por sí sola, ¿implica que el modelo es útil? Considera el coste de un falso negativo: una factura que requería revisión y que el modelo dejó pasar. Si reducir esos FN importa más que el coste de más revisiones manuales, ¿qué métrica y qué umbral deberíamos priorizar en el siguiente modelo?